# ML workflow — data from MongoDB only

The ETL writes **silver** collections; this notebook reads them over the network.
No MinIO or API keys required — set `MONGODB_HOST` / `MONGODB_PORT` (or `MONGODB_URI`).

| Where Jupyter runs | `MONGODB_HOST` |
|--------------------|----------------|
| Docker `jupyter` service | `mongodb` |
| Your PC (Mongo in Docker) | `localhost` and port `27018` |

In [1]:
import sys
from pathlib import Path

ROOT = Path("/app") if Path("/app/MachineLearning").exists() else Path.cwd().parent.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from MachineLearning.mongo_config import MongoConfig
from MachineLearning.mongo_loader import get_collection_counts, load_merged_dataset, verify_connection
from MachineLearning.data_pipeline import build_all
from MachineLearning.train_aqi import train

cfg = MongoConfig.from_env()
print("Target:", cfg.describe())
verify_connection(cfg)
print("Collections:", get_collection_counts(cfg))

Target: mongodb:27017/weather_etl as weather_user
Collections: {'silver_weather': 20, 'silver_air_quality': 20, 'gold_weather_daily': 20, 'gold_air_quality_daily': 20, 'gold_daily': 20}


In [2]:
df = load_merged_dataset(config=cfg)
df.head()

,city,date_paris,hour_paris,timestamp_paris_wx,temp,humidity,pressure,wind_speed,wind_gust,clouds,weather_main,timestamp_paris_aq,aqi,pm25,pm10,no2,o3,alert_level,datetime
0,bordeaux,2026-05-19,23,2026-05-19T23:00:00+02:00,16.24,83,1023,2.29,5.62,0,clear,2026-05-19T23:00:00+02:00,25,25.0,9,9.2,3.3,good,2026-05-19 23:00:00+02:00
1,bordeaux,2026-05-20,0,2026-05-20T00:00:00+02:00,16.08,84,1023,2.39,4.73,0,clear,2026-05-20T00:00:00+02:00,21,21.0,19,9.6,3.3,good,2026-05-20 00:00:00+02:00
2,lille,2026-05-19,23,2026-05-19T23:00:00+02:00,13.94,94,1014,5.26,11.05,100,rain,2026-05-19T23:00:00+02:00,25,25.0,6,10.1,15.6,good,2026-05-19 23:00:00+02:00
3,lille,2026-05-20,0,2026-05-20T00:00:00+02:00,13.04,91,1015,4.74,9.72,75,rain,2026-05-20T00:00:00+02:00,25,25.0,6,10.1,15.6,good,2026-05-20 00:00:00+02:00
4,lyon,2026-05-19,23,2026-05-19T23:00:00+02:00,16.80,75,1021,1.40,1.47,0,clear,2026-05-19T23:00:00+02:00,20,NaN,20,10.1,NaN,good,2026-05-19 23:00:00+02:00


In [3]:
silver, gold = build_all(config=cfg)
gold.describe()

Silver dataset: 20 rows -> /app/MachineLearning/data/silver/dataset_silver.csv
Gold dataset: 20 rows -> /app/MachineLearning/data/gold/dataset_gold.csv


,city_code,temp,humidity,pressure,wind_speed,hour,month,aqi
count,20.000000,20.000000,20.000000,20.00000,20.000000,20.000000,20.0,20.000000
mean,4.500000,16.140500,78.900000,1019.90000,3.051000,11.500000,5.0,28.550000
std,2.946898,1.756211,10.020505,2.44734,1.319182,11.798751,0.0,11.385471
min,0.000000,13.040000,56.000000,1014.00000,1.140000,0.000000,5.0,12.000000
25%,2.000000,15.427500,74.250000,1019.00000,1.765000,0.000000,5.0,23.500000
50%,4.500000,15.940000,81.000000,1021.00000,3.190000,11.500000,5.0,26.500000
75%,7.000000,16.380000,86.250000,1021.00000,3.872500,23.000000,5.0,30.250000
max,9.000000,20.970000,94.000000,1023.00000,5.260000,23.000000,5.0,57.000000


In [4]:
train()

Dataset loaded: 20 rows
MAE: 8.30
Model saved -> /app/MachineLearning/models/aqi_model.pkl


In [5]:
from MachineLearning.predict import predict_aqi

predicted = predict_aqi({
    "city_code": 0,      # paris
    "temp": 12.0,
    "humidity": 60.0,
    "pressure": 1015.0,
    "wind_speed": 3.0,
    "hour": 14,
    "month": 5,
})
print(f"Predicted AQI: {predicted:.0f}")

Predicted AQI: 34


In [6]:
from MachineLearning.data_loader import load_dataset
from MachineLearning.predict import predict_aqi
from config.towns import FRENCH_TOWNS

city_codes = {t.name: i for i, t in enumerate(FRENCH_TOWNS)}

df = load_dataset()
row = df.iloc[-1]  # latest merged row

predict_aqi({
    "city_code": city_codes[row["city"]],
    "temp": row["temp"],
    "humidity": row["humidity"],
    "pressure": row["pressure"],
    "wind_speed": row["wind_speed"],
    "hour": int(row["hour_paris"]),
    "month": int(row["datetime"].month),
})

27.34